In [1]:
import pandas as pd

df = pd.read_csv("credit_applicants.csv")
df.head()

,applicant_id,age,monthly_income_inr,existing_loans_count,credit_utilization_ratio,upi_monthly_inflow_inr,bounced_payments_count,credit_bureau_score,employment_type,default
0,APP1000,59,41646,4,0.80,17398,2,NaN,salaried,1
1,APP1001,49,119185,0,0.08,58503,3,380.0,salaried,0
2,APP1002,35,38049,0,0.59,8638,1,788.0,salaried,1
3,APP1003,28,113116,0,0.26,8570,2,387.0,salaried,0
4,APP1004,41,112379,2,0.16,70785,3,493.0,salaried,0


In [2]:
default_rate = df["default"].mean()
missing_pct = df["credit_bureau_score"].isna().mean() * 100

print(f"Default rate: {default_rate:.4f} ({default_rate*100:.2f}%)")
print(f"Missing credit_bureau_score: {missing_pct:.2f}%")

Default rate: 0.2025 (20.25%)
Missing credit_bureau_score: 20.00%


In [3]:
df["is_thin_file"] = df["credit_bureau_score"].isna().astype(int)
df[["applicant_id", "credit_bureau_score", "is_thin_file"]].head(10)

,applicant_id,credit_bureau_score,is_thin_file
0,APP1000,NaN,1
1,APP1001,380.0,0
2,APP1002,788.0,0
3,APP1003,387.0,0
4,APP1004,493.0,0
5,APP1005,403.0,0
6,APP1006,395.0,0
7,APP1007,722.0,0
8,APP1008,NaN,1
9,APP1009,NaN,1


In [4]:
from sklearn.model_selection import train_test_split

feature_cols = ["age", "monthly_income_inr", "existing_loans_count",
                 "credit_utilization_ratio", "upi_monthly_inflow_inr",
                 "bounced_payments_count", "credit_bureau_score",
                 "employment_type", "is_thin_file"]

X = df[feature_cols]
y = df["default"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)

print("Training set size:", X_train.shape[0])
print("Test set size:", X_test.shape[0])
print("Training default rate:", y_train.mean())
print("Test default rate:", y_test.mean())

Training set size: 300
Test set size: 100
Training default rate: 0.20333333333333334
Test default rate: 0.2


In [5]:
train_median = X_train["credit_bureau_score"].median()
print("Median credit_bureau_score from TRAINING split only:", train_median)

X_train["credit_bureau_score"] = X_train["credit_bureau_score"].fillna(train_median)
X_test["credit_bureau_score"] = X_test["credit_bureau_score"].fillna(train_median)

print("Missing values remaining in X_train:", X_train["credit_bureau_score"].isna().sum())
print("Missing values remaining in X_test:", X_test["credit_bureau_score"].isna().sum())

Median credit_bureau_score from TRAINING split only: 612.0
Missing values remaining in X_train: 0
Missing values remaining in X_test: 0


In [6]:
X_train = pd.get_dummies(X_train, columns=["employment_type"], drop_first=False)
X_test = pd.get_dummies(X_test, columns=["employment_type"], drop_first=False)

X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

X_train.head()

,age,monthly_income_inr,existing_loans_count,credit_utilization_ratio,upi_monthly_inflow_inr,bounced_payments_count,credit_bureau_score,is_thin_file,employment_type_gig,employment_type_salaried,employment_type_self_employed
84,28,117795,2,0.53,98652,1,582.0,0,False,False,True
10,44,24823,1,0.33,24911,1,612.0,1,False,True,False
384,49,16828,1,0.10,67061,0,624.0,0,False,False,True
359,39,91323,2,0.92,41891,1,808.0,0,True,False,False
11,56,56975,4,0.54,106061,1,677.0,0,False,True,False


In [7]:
print(X_train.columns.tolist())

['age', 'monthly_income_inr', 'existing_loans_count', 'credit_utilization_ratio', 'upi_monthly_inflow_inr', 'bounced_payments_count', 'credit_bureau_score', 'is_thin_file', 'employment_type_gig', 'employment_type_salaried', 'employment_type_self_employed']


In [8]:
from sklearn.preprocessing import StandardScaler

numeric_cols = ["age", "monthly_income_inr", "existing_loans_count",
                 "credit_utilization_ratio", "upi_monthly_inflow_inr",
                 "bounced_payments_count", "credit_bureau_score"]

scaler = StandardScaler()

X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test[numeric_cols] = scaler.transform(X_test[numeric_cols])

X_train[numeric_cols].describe()

,age,monthly_income_inr,existing_loans_count,credit_utilization_ratio,upi_monthly_inflow_inr,bounced_payments_count,credit_bureau_score
count,3.000000e+02,3.000000e+02,3.000000e+02,300.000000,3.000000e+02,3.000000e+02,3.000000e+02
mean,2.013204e-16,1.539509e-16,-2.960595e-17,0.000000,-7.401487e-17,-7.105427e-17,3.256654e-16
std,1.001671e+00,1.001671e+00,1.001671e+00,1.001671,1.001671e+00,1.001671e+00,1.001671e+00
min,-1.765572e+00,-1.622322e+00,-1.336554e+00,-1.620535,-1.812288e+00,-1.127427e+00,-1.990609e+00
25%,-7.915853e-01,-9.500497e-01,-6.416388e-01,-0.882922,-8.735946e-01,-1.127427e+00,-6.895393e-01
50%,1.824011e-01,2.110522e-02,5.327687e-02,0.002213,4.995835e-02,-2.278843e-01,4.899433e-02
75%,8.022106e-01,9.361945e-01,7.481925e-01,0.887348,9.129417e-01,6.716589e-01,7.019962e-01
max,1.599109e+00,1.754419e+00,1.443108e+00,1.698722,1.613557e+00,3.370288e+00,1.930692e+00


In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

log_reg = LogisticRegression(random_state=42)
log_reg.fit(X_train, y_train)

dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train, y_train)

print("Both models trained successfully.")

Both models trained successfully.


In [10]:
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

def evaluate_model(model, name):
    preds = model.predict(X_test)
    probs = model.predict_proba(X_test)[:, 1]

    print(f"--- {name} ---")
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, preds))
    print("Accuracy:", accuracy_score(y_test, preds))
    print("Precision:", precision_score(y_test, preds))
    print("Recall:", recall_score(y_test, preds))
    print("F1 Score:", f1_score(y_test, preds))
    print("ROC-AUC:", roc_auc_score(y_test, probs))
    print()

evaluate_model(log_reg, "Logistic Regression")
evaluate_model(dt, "Decision Tree")

--- Logistic Regression ---
Confusion Matrix:
[[69 11]
 [13  7]]
Accuracy: 0.76
Precision: 0.3888888888888889
Recall: 0.35
F1 Score: 0.3684210526315789
ROC-AUC: 0.71875

--- Decision Tree ---
Confusion Matrix:
[[61 19]
 [14  6]]
Accuracy: 0.67
Precision: 0.24
Recall: 0.3
F1 Score: 0.26666666666666666
ROC-AUC: 0.53125



In [11]:
test_probs = log_reg.predict_proba(X_test)[:, 1]

pricing_df = X_test.copy()
pricing_df["predicted_default_prob"] = test_probs
pricing_df["actual_default"] = y_test.values

pricing_df["risk_tier"] = pd.qcut(pricing_df["predicted_default_prob"], q=4,
                                    labels=["Tier 1 (Low Risk)", "Tier 2 (Medium)",
                                            "Tier 3 (High)", "Tier 4 (Very High)"])

rate_map = {
    "Tier 1 (Low Risk)": 14,
    "Tier 2 (Medium)": 20,
    "Tier 3 (High)": 28,
    "Tier 4 (Very High)": 36,
}
pricing_df["interest_rate_pct"] = pricing_df["risk_tier"].map(rate_map)

pricing_summary = pricing_df.groupby("risk_tier").agg(
    num_applicants=("actual_default", "count"),
    observed_default_rate=("actual_default", "mean"),
    assigned_interest_rate_pct=("interest_rate_pct", "first")
)
print(pricing_summary)

C:\Users\sidra hasan\AppData\Local\Temp\ipykernel_12824\2735349772.py:19: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  pricing_summary = pricing_df.groupby("risk_tier").agg(


                    num_applicants  observed_default_rate  \
risk_tier                                                   
Tier 1 (Low Risk)               25                   0.08   
Tier 2 (Medium)                 25                   0.12   
Tier 3 (High)                   25                   0.20   
Tier 4 (Very High)              25                   0.40   

                   assigned_interest_rate_pct  
risk_tier                                      
Tier 1 (Low Risk)                          14  
Tier 2 (Medium)                            20  
Tier 3 (High)                              28  
Tier 4 (Very High)                         36  


In [12]:
behaviour_df = pd.read_csv("txn_behaviour.csv")
print(behaviour_df.shape)
behaviour_df.head()

(265, 6)


,txn_id,applicant_id,txn_hour,is_new_device,txn_amount_inr,channel
0,BTXN5000,APP1126,15,0,199,P2P
1,BTXN5001,APP1167,22,0,499,P2P
2,BTXN5002,APP1079,18,0,1999,P2P
3,BTXN5003,APP1293,18,1,999,P2M
4,BTXN5004,APP1332,12,0,199,P2P


In [13]:
from sklearn.ensemble import IsolationForest

anomaly_features = ["txn_hour", "is_new_device", "txn_amount_inr"]
X_anomaly = behaviour_df[anomaly_features]

anomaly_scaler = StandardScaler()
X_anomaly_scaled = anomaly_scaler.fit_transform(X_anomaly)

contamination_rate = 15 / 265
print("Contamination rate:", contamination_rate)

iso_forest = IsolationForest(random_state=42, contamination=contamination_rate)
behaviour_df["anomaly_prediction"] = iso_forest.fit_predict(X_anomaly_scaled)

behaviour_df["anomaly_prediction"].value_counts()

Contamination rate: 0.05660377358490566


anomaly_prediction
 1    250
-1     15
Name: count, dtype: int64

In [14]:
behaviour_df["is_true_anomaly"] = behaviour_df["txn_id"].str.startswith("BTXNA")

true_anomalies_caught = behaviour_df[
    (behaviour_df["is_true_anomaly"] == True) & (behaviour_df["anomaly_prediction"] == -1)
]

print("Total true (seeded) anomalies:", behaviour_df["is_true_anomaly"].sum())
print("True anomalies caught by Isolation Forest:", len(true_anomalies_caught))
print("Recall:", len(true_anomalies_caught) / behaviour_df["is_true_anomaly"].sum())

Total true (seeded) anomalies: 15
True anomalies caught by Isolation Forest: 11
Recall: 0.7333333333333333


In [15]:
final_comparison = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1 Score", "ROC-AUC"],
    "Logistic Regression": [0.76, 0.39, 0.35, 0.37, 0.72],
    "Decision Tree": [0.67, 0.24, 0.30, 0.27, 0.53],
})
print(final_comparison)
print()
print("Isolation Forest recall on seeded fraud anomalies: 11/15 = 73.33%")

      Metric  Logistic Regression  Decision Tree
0   Accuracy                 0.76           0.67
1  Precision                 0.39           0.24
2     Recall                 0.35           0.30
3   F1 Score                 0.37           0.27
4    ROC-AUC                 0.72           0.53

Isolation Forest recall on seeded fraud anomalies: 11/15 = 73.33%
